In [ ]:
# NLP Project 2: Movie Review Intelligence System (IMDb)
# Shahd Batta, Raneem Amer - Supervisor: Dr. Ahmed Fahil
# Run in Google Colab with a GPU: Runtime > Change runtime type > T4 GPU
# Slow parts are cached on Google Drive (embeddings, DistilBERT predictions),
# so later runs take a few minutes instead of ~15.

!pip install -q sentence-transformers gradio

# 0. Setup
import os, re, json, time
from functools import lru_cache
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import nltk
import torch

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay)

SEED = 42
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 60)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("GPU available:", torch.cuda.is_available(), "| device:", device)

# Same project folder as Project 1 (data/ and models/ come from Phase 1)
USE_DRIVE = True
PROJECT_DIR = "/content/drive/MyDrive/NLP_Project"
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        os.makedirs(PROJECT_DIR, exist_ok=True)
        os.chdir(PROJECT_DIR)
    except Exception as e:
        print("Google Drive not used:", e)

for folder in ["data", "models", "results_p2", "embeddings"]:
    os.makedirs(folder, exist_ok=True)
print("Working directory:", os.getcwd())

# 1. Project 1 baseline (same data, same split, same model)
TRAIN_CSV, TEST_CSV = "data/imdb_train.csv", "data/imdb_test.csv"
if os.path.exists(TRAIN_CSV) and os.path.exists(TEST_CSV):
    train_df, test_df = pd.read_csv(TRAIN_CSV), pd.read_csv(TEST_CSV)
    print("\nLoaded the saved Project 1 split from data/")
else:
    from datasets import load_dataset
    ds = load_dataset("stanfordnlp/imdb")
    train_df, test_df = ds["train"].to_pandas(), ds["test"].to_pandas()
    train_df.insert(0, "id", [f"train_{i}" for i in range(len(train_df))])
    test_df.insert(0, "id", [f"test_{i}" for i in range(len(test_df))])
    train_df[["id", "text", "label"]].to_csv(TRAIN_CSV, index=False)
    test_df[["id", "text", "label"]].to_csv(TEST_CSV, index=False)
    print("\nDownloaded the official IMDb split from Hugging Face")
print("Train:", train_df.shape, "| Test:", test_df.shape, "|", test_df["label"].value_counts().to_dict())
y_train, y_test = train_df["label"].values, test_df["label"].values

# ----- Project 1 preprocessing (unchanged) -----
for r in ["stopwords", "wordnet", "omw-1.4"]:
    nltk.download(r, quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

negations = {"not", "no", "nor", "never", "none", "nothing", "nobody", "neither", "without"}
STOP_WORDS = set(stopwords.words("english")) - negations
lemmatizer = WordNetLemmatizer()

def convert_ratings(text):
    def replace(match):
        score = float(match.group(1))
        if score <= 4:
            return " ratingbad "
        elif score <= 6:
            return " ratingmid "
        return " ratinggood "
    return re.sub(r"\b(10|\d(?:\.\d)?)\s*/\s*10\b", replace, text)

@lru_cache(maxsize=None)
def lemmatize_word(word):
    lemma = lemmatizer.lemmatize(word, pos="v")
    if lemma == word:
        lemma = lemmatizer.lemmatize(word, pos="n")
    return lemma

def clean_text(text):
    text = re.sub(r"<[^>]+>", " ", text)
    text = convert_ratings(text)
    text = text.lower()
    text = text.replace("won't", "will not").replace("can't", "can not")
    text = re.sub(r"n't", " not", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def preprocess(text):
    tokens = clean_text(text).split()
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 1]
    return " ".join(lemmatize_word(t) for t in tokens)

print("Preprocessing check:", preprocess("I didn't like this movie!<br /><br />The acting was NOT good... 2/10"))

train_df["clean"] = train_df["text"].apply(preprocess)
test_df["clean"] = test_df["text"].apply(preprocess)

# ----- Load the saved Project 1 model (retrain it with the same settings if missing) -----
if os.path.exists("models/bow_vectorizer.joblib") and os.path.exists("models/sentiment_classifier.joblib"):
    vectorizer = joblib.load("models/bow_vectorizer.joblib")
    classifier = joblib.load("models/sentiment_classifier.joblib")
    print("Loaded the saved Project 1 model from models/")
else:
    vectorizer = CountVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.95)
    classifier = LogisticRegression(C=0.5, max_iter=2000, solver="liblinear").fit(
        vectorizer.fit_transform(train_df["clean"]), y_train)
    joblib.dump(vectorizer, "models/bow_vectorizer.joblib")
    joblib.dump(classifier, "models/sentiment_classifier.joblib")
    print("Retrained the Project 1 model with its original settings")

X_test_bow = vectorizer.transform(test_df["clean"])
p1_pred = classifier.predict(X_test_bow)
p1_proba = classifier.predict_proba(X_test_bow)[:, 1]

def metrics_row(name, rep, pred):
    return {"Version": name, "Representation": rep,
            "Accuracy": round(accuracy_score(y_test, pred), 4),
            "Precision": round(precision_score(y_test, pred), 4),
            "Recall": round(recall_score(y_test, pred), 4),
            "F1": round(f1_score(y_test, pred), 4)}

baseline = metrics_row("Project 1 baseline", "BoW + Logistic Regression", p1_pred)
with open("results_p2/baseline_p1.json", "w") as f:
    json.dump(baseline, f, indent=2)
print("\nPROJECT 1 BASELINE (re-run):", baseline)
print("Confusion matrix:\n", confusion_matrix(y_test, p1_pred))

# 2. Prepare the same data for modern NLP
# Change: modern models need natural text, so we only remove HTML tags.
# Stop words, punctuation and word forms are kept.
def light_clean(text):
    text = re.sub(r"<[^>]+>", " ", text)
    return re.sub(r"\s+", " ", text).strip()

train_df["text_p2"] = train_df["text"].apply(light_clean)
test_df["text_p2"] = test_df["text"].apply(light_clean)
print("\nProject 1 input:", preprocess(test_df["text"][0])[:150])
print("Project 2 input:", test_df["text_p2"][0][:150])

# 3. Embeddings (Sentence Transformers, all-MiniLM-L6-v2)
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
print("\nEmbedding size:", embedder.get_sentence_embedding_dimension(), "| max tokens:", embedder.max_seq_length)

# ----- Related meanings get similar vectors (compared with Bag of Words) -----
pairs = [
    ("The film was boring.", "I fell asleep halfway through."),
    ("The acting was superb.", "Brilliant performances from the whole cast."),
    ("A complete waste of money.", "I want my ticket refund."),
    ("The film was boring.", "The acting was superb."),
    ("The soundtrack was beautiful.", "The tax law changed last year."),
]
rows = []
for a, b in pairs:
    ea, eb = embedder.encode([a, b])
    bow = CountVectorizer().fit([a, b])
    va, vb = bow.transform([a, b])
    rows.append({"Text A": a, "Text B": b,
                 "BoW similarity": round(cosine_similarity(va, vb)[0, 0], 2),
                 "Embedding similarity": round(float(cosine_similarity([ea], [eb])[0, 0]), 2)})
sim_df = pd.DataFrame(rows)
sim_df.to_csv("results_p2/similarity_examples.csv", index=False)
print(sim_df.to_string(index=False))

# ----- Embeddings for all reviews (cached on Drive) -----
def get_embeddings(texts, path):
    if os.path.exists(path):
        return np.load(path)
    start = time.time()
    emb = embedder.encode(list(texts), batch_size=64, show_progress_bar=True,
                          normalize_embeddings=True, convert_to_numpy=True)
    np.save(path, emb)
    print(f"Encoded in {(time.time() - start) / 60:.1f} min")
    return emb

train_emb = get_embeddings(train_df["text_p2"], "embeddings/train_emb.npy")
test_emb = get_embeddings(test_df["text_p2"], "embeddings/test_emb.npy")
print("Train embeddings:", train_emb.shape, "| Test embeddings:", test_emb.shape)

# 4. Semantic similarity / semantic search (cosine similarity)
def semantic_search(query, k=5, corpus_emb=train_emb, corpus_df=train_df):
    q = embedder.encode([query], normalize_embeddings=True)[0]
    scores = corpus_emb @ q                       # vectors are normalized: dot product = cosine
    top = np.argsort(-scores)[:k]
    return pd.DataFrame({
        "rank": range(1, k + 1),
        "id": corpus_df["id"].values[top],
        "similarity": scores[top].round(3),
        "label": ["Positive" if l == 1 else "Negative" for l in corpus_df["label"].values[top]],
        "review": [t[:150] + "..." for t in corpus_df["text_p2"].values[top]],
    })

queries = [
    "a movie that put me to sleep",
    "beautiful soundtrack and amazing music",
    "the plot made no sense at all",
    "a scary horror film that kept me awake at night",
    "a perfect film to watch with kids and family",
    "the special effects looked cheap and fake",
    "I laughed from start to finish",
]
all_results = []
for q in queries:
    res = semantic_search(q, k=5)
    res.insert(0, "query", q)
    all_results.append(res)
    print("\n" + "=" * 100 + "\nQUERY:", q)
    print(res.drop(columns="query").to_string(index=False))
pd.concat(all_results).to_csv("results_p2/semantic_search_examples.csv", index=False)

# ----- Comparison with keyword search (TF-IDF) -----
tfidf = TfidfVectorizer(stop_words="english", min_df=3)
train_tfidf = tfidf.fit_transform(train_df["text_p2"])

def keyword_search(query, k=5):
    scores = (train_tfidf @ tfidf.transform([query]).T).toarray().ravel()
    top = np.argsort(-scores)[:k]
    return pd.DataFrame({"rank": range(1, k + 1), "similarity": scores[top].round(3),
                         "review": [t[:150] + "..." for t in train_df["text_p2"].values[top]]})

q = "a movie that put me to sleep"
print("\nKEYWORD SEARCH (TF-IDF):\n", keyword_search(q).to_string(index=False))
print("\nSEMANTIC SEARCH (embeddings):\n", semantic_search(q).drop(columns=["id", "label"]).to_string(index=False))

# 5. Modern classification
# Same validation split as Project 1 (10%, stratified, random_state=42)
idx_tr, idx_val = train_test_split(np.arange(len(train_df)), test_size=0.10,
                                   stratify=y_train, random_state=SEED)
y_val = y_train[idx_val]

# ----- (a) Embeddings + Logistic Regression -----
print("\n(a) Choosing C on the validation set:")
best_C, best_acc = None, 0
for C in [0.1, 1, 10]:
    m = LogisticRegression(C=C, max_iter=2000).fit(train_emb[idx_tr], y_train[idx_tr])
    acc = accuracy_score(y_val, m.predict(train_emb[idx_val]))
    print(f"  C = {C:<4} -> validation accuracy = {acc:.4f}")
    if acc > best_acc:
        best_C, best_acc = C, acc
for C in [100, 1000]:   # extra check: larger C does not help
    m = LogisticRegression(C=C, max_iter=5000).fit(train_emb[idx_tr], y_train[idx_tr])
    print(f"  C = {C:<4} -> validation accuracy = {accuracy_score(y_val, m.predict(train_emb[idx_val])):.4f} (check only)")

emb_clf = LogisticRegression(C=best_C, max_iter=2000).fit(train_emb, y_train)
emb_pred = emb_clf.predict(test_emb)

# ----- (b) Pretrained Transformer: DistilBERT fine-tuned on SST-2 (no training by us) -----
from transformers import pipeline
sentiment_model = pipeline("sentiment-analysis",
                           model="distilbert-base-uncased-finetuned-sst-2-english",
                           device=0 if torch.cuda.is_available() else -1)
print("\nDistilBERT on sarcasm:", sentiment_model("Just brilliant. Another sequel nobody asked for."))

def to_pos_proba(outputs):
    return np.array([o["score"] if o["label"] == "POSITIVE" else 1 - o["score"] for o in outputs])

TF_PATH = "results_p2/transformer_test_proba.npy"
if os.path.exists(TF_PATH):
    tf_proba = np.load(TF_PATH)
else:
    start = time.time()
    tf_proba = to_pos_proba(sentiment_model(list(test_df["text_p2"]), batch_size=64,
                                            truncation=True, max_length=512))
    np.save(TF_PATH, tf_proba)
    print(f"DistilBERT on the test set: {(time.time() - start) / 60:.1f} min")
tf_pred = (tf_proba >= 0.5).astype(int)

# ----- (c) Ensemble: weight chosen on the validation set -----
vec_val = CountVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.95)
p1_val_model = LogisticRegression(C=0.5, max_iter=2000, solver="liblinear").fit(
    vec_val.fit_transform(train_df["clean"].values[idx_tr]), y_train[idx_tr])
p1_val_proba = p1_val_model.predict_proba(vec_val.transform(train_df["clean"].values[idx_val]))[:, 1]
tf_val_proba = to_pos_proba(sentiment_model(list(train_df["text_p2"].values[idx_val]), batch_size=64,
                                            truncation=True, max_length=512))

print("\n(c) DistilBERT weight -> validation accuracy")
best_w, best_acc = 0, 0
for w in np.arange(0, 1.01, 0.1):
    acc = accuracy_score(y_val, (w * tf_val_proba + (1 - w) * p1_val_proba >= 0.5).astype(int))
    print(f"  w = {w:.1f} -> {acc:.4f}")
    if acc > best_acc:
        best_w, best_acc = round(w, 1), acc
ens_proba = best_w * tf_proba + (1 - best_w) * p1_proba
ens_pred = (ens_proba >= 0.5).astype(int)

comparison = pd.DataFrame([
    baseline,
    metrics_row("Project 2 (a)", "MiniLM embeddings + Logistic Regression", emb_pred),
    metrics_row("Project 2 (b)", "DistilBERT (SST-2), no training", tf_pred),
    metrics_row("Project 2 (c)", f"Ensemble: {best_w} x DistilBERT + {round(1 - best_w, 1)} x BoW", ens_pred),
])
comparison.to_csv("results_p2/comparison_table.csv", index=False)
print("\nPROJECT 1 vs PROJECT 2 (25,000 test reviews):\n", comparison.to_string(index=False))

# 6. Advanced NLP feature: aspect-based sentiment
# (acting, story, music, directing, visuals)
ASPECTS = {
    "Acting":    {"keywords": ["acting", "actor", "actors", "actress", "actresses", "performance", "performances",
                               "cast", "role", "roles", "portrayal", "portrayed", "played"],
                  "examples": ["The actors were excellent.", "She gave a brilliant performance.",
                               "He was outstanding in the lead.", "The cast was terrible and wooden."]},
    "Story":     {"keywords": ["story", "plot", "script", "storyline", "ending", "twist", "narrative",
                               "screenplay", "dialogue", "writing", "written"],
                  "examples": ["The story was engaging.", "The plot made no sense.",
                               "The ending was a big letdown.", "Nothing really happens in this film."]},
    "Music":     {"keywords": ["music", "soundtrack", "score", "song", "songs", "composer"],
                  "examples": ["The music was beautiful.", "Great soundtrack.", "The songs were catchy.",
                               "The theme tune stays in your head."]},
    "Directing": {"keywords": ["director", "directed", "directing", "direction", "pacing", "paced", "editing"],
                  "examples": ["The director did a great job.", "The pace was too slow.",
                               "It drags on and feels far too long."]},
    "Visuals":   {"keywords": ["visual", "visuals", "effects", "cinematography", "photography", "camera",
                               "scenery", "cgi", "animation", "shots"],
                  "examples": ["The special effects were impressive.", "Beautiful cinematography.",
                               "Every shot is gorgeous to look at.", "The effects looked cheap."]},
}
aspect_names = list(ASPECTS)
proto_text, proto_aspect = [], []
for a in aspect_names:
    for ex in ASPECTS[a]["examples"]:
        proto_text.append(ex)
        proto_aspect.append(a)
proto_emb = embedder.encode(proto_text, normalize_embeddings=True)
proto_aspect = np.array(proto_aspect)
SIM_THRESHOLD = 0.45

SPLIT_RE = re.compile(
    r"(?<=[.!?])\)?\s+"                                              # end of sentence, even before ")"
    r"|;\s+"                                                         # semicolon
    r"|,?\s+\b(?:but|yet|however|although|though|whereas)\b,?\s+",   # contrast words
    flags=re.IGNORECASE)

def split_sentences(text):
    parts = SPLIT_RE.split(light_clean(text))
    return [p.strip() for p in parts if p and len(p.split()) >= 2]

def detect_aspects(sentence, sent_emb):
    words = set(re.findall(r"[a-z]+", sentence.lower()))
    found = [(a, "keyword") for a in aspect_names if words & set(ASPECTS[a]["keywords"])]
    if found:
        return found
    sims = proto_emb @ sent_emb
    best = {a: sims[proto_aspect == a].max() for a in aspect_names}
    a_top = max(best, key=best.get)
    if best[a_top] >= SIM_THRESHOLD:
        return [(a_top, f"embedding ({best[a_top]:.2f})")]
    return []

# ----- Threshold check -----
check = [
    ("Meryl Streep was outstanding.", "Acting"), ("Tom Hanks was perfect as the father.", "Acting"),
    ("It drags on for far too long.", "Directing"), ("I could not stop humming the theme afterwards.", "Music"),
    ("Every frame looks like a painting.", "Visuals"), ("Nothing makes sense in the last twenty minutes.", "Story"),
    ("I watched it with my sister last weekend.", "None"), ("I bought the DVD at a supermarket.", "None"),
    ("This is the third film in the series.", "None"),
]
rows = []
for s, expected in check:
    sims = proto_emb @ embedder.encode([s], normalize_embeddings=True)[0]
    best = {a: sims[proto_aspect == a].max() for a in aspect_names}
    a_top = max(best, key=best.get)
    rows.append({"sentence": s, "expected": expected, "closest": a_top, "similarity": round(float(best[a_top]), 2)})
print("\nTHRESHOLD CHECK:\n", pd.DataFrame(rows).to_string(index=False))

def analyze_aspects(review):
    sentences = split_sentences(review)
    if not sentences:
        return pd.DataFrame()
    sent_embs = embedder.encode(sentences, normalize_embeddings=True)
    rows = [{"aspect": a, "sentence": s, "found by": how}
            for s, e in zip(sentences, sent_embs) for a, how in detect_aspects(s, e)]
    if not rows:
        return pd.DataFrame()
    found = pd.DataFrame(rows)
    outs = sentiment_model(list(found["sentence"]), truncation=True, max_length=512)
    found["opinion"] = ["Positive" if o["label"] == "POSITIVE" else "Negative" for o in outs]
    found["confidence"] = [round(o["score"], 3) for o in outs]
    return found

for t in ["Great cast, beautiful visuals, but the story was boring and way too long.",
          "The soundtrack is gorgeous and the cinematography is stunning. Sadly the plot makes no sense and the ending is rushed.",
          "Meryl Streep was outstanding. It drags on for far too long though."]:
    print("\n" + "=" * 100 + "\nREVIEW:", t)
    print(analyze_aspects(t).to_string(index=False))

# ----- Aspect analysis on 1,000 test reviews -----
sample = test_df.sample(1000, random_state=SEED).reset_index(drop=True)
sents = pd.DataFrame([{"review": i, "review_label": r["label"], "sentence": s}
                      for i, r in sample.iterrows() for s in split_sentences(r["text"])])
embs = embedder.encode(list(sents["sentence"]), batch_size=128, normalize_embeddings=True)
asp = pd.DataFrame([{**r.to_dict(), "aspect": a, "found_by": "keyword" if how == "keyword" else "embedding"}
                    for (_, r), e in zip(sents.iterrows(), embs) for a, how in detect_aspects(r["sentence"], e)])
asp["opinion"] = [1 if o["label"] == "POSITIVE" else 0
                  for o in sentiment_model(list(asp["sentence"]), batch_size=64, truncation=True, max_length=512)]
asp.to_csv("results_p2/aspect_sentences_sample.csv", index=False)
print(f"\nSentences: {len(sents):,} | with an aspect: {len(asp):,}")

summary = asp.groupby("aspect").agg(
    reviews_mentioning=("review", "nunique"),
    found_by_embedding=("found_by", lambda x: f"{100 * (x == 'embedding').mean():.0f}%")).reset_index()
pos_in_pos = asp[asp.review_label == 1].groupby("aspect")["opinion"].mean()
pos_in_neg = asp[asp.review_label == 0].groupby("aspect")["opinion"].mean()
summary["% positive (positive reviews)"] = summary["aspect"].map(lambda a: f"{100 * pos_in_pos.get(a, 0):.0f}%")
summary["% positive (negative reviews)"] = summary["aspect"].map(lambda a: f"{100 * pos_in_neg.get(a, 0):.0f}%")
summary.to_csv("results_p2/aspect_summary.csv", index=False)
print(summary.to_string(index=False))

# ----- Sample for the manual check (10 keyword + 10 embedding sentences) -----
manual = pd.concat([asp[asp.found_by == "keyword"].sample(10, random_state=SEED),
                    asp[asp.found_by == "embedding"].sample(10, random_state=SEED)]).reset_index(drop=True)
manual["opinion"] = manual["opinion"].map({1: "Positive", 0: "Negative"})
manual[["found_by", "aspect", "opinion", "sentence"]].to_csv("results_p2/aspect_manual_check.csv")
print("\nMANUAL CHECK SAMPLE:")
for i, r in manual.iterrows():
    print(f"{i + 1:2d}. [{r['found_by']:9s}] {r['aspect']:9s} | {r['opinion']:8s} | {r['sentence'][:160]}")

# 7. Project 1 vs Project 2: errors and examples
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, pred, title in [(axes[0], p1_pred, f"Project 1: BoW + LR\naccuracy {accuracy_score(y_test, p1_pred):.4f}"),
                        (axes[1], ens_pred, f"Project 2: Ensemble\naccuracy {accuracy_score(y_test, ens_pred):.4f}")]:
    ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=["Negative", "Positive"]).plot(
        ax=ax, cmap="Blues", values_format=",", colorbar=False)
    ax.set_title(title)
plt.tight_layout()
plt.savefig("results_p2/confusion_matrices_p1_vs_p2.png", dpi=120)
plt.show()

fixed = (p1_pred != y_test) & (ens_pred == y_test)
broken = (p1_pred == y_test) & (ens_pred != y_test)
print(f"Fixed by Project 2: {fixed.sum():,} | broken: {broken.sum():,} | net: {fixed.sum() - broken.sum():+,}")

test_df["n_words"] = test_df["text_p2"].str.split().str.len()

def show(rows, title):
    print("\n" + "=" * 100 + "\n" + title + "\n" + "=" * 100)
    for i in rows:
        r = test_df.loc[i]
        print(f"\n[{r['id']}] TRUE: {'Positive' if y_test[i] else 'Negative'}")
        print(f"  P1 BoW:     {'Positive' if p1_pred[i] else 'Negative'} (P(pos) = {p1_proba[i]:.2f})")
        print(f"  DistilBERT: P(pos) = {tf_proba[i]:.2f}")
        print(f"  Ensemble:   {'Positive' if ens_pred[i] else 'Negative'} (P(pos) = {ens_proba[i]:.2f})")
        print("  ", r["text_p2"][:500], "..." if len(r["text_p2"]) > 500 else "")

cand = np.where(fixed & (test_df["n_words"].values < 130))[0]
cand = cand[np.argsort(-np.abs(ens_proba[cand] - 0.5))][:3]
show(cand, "IMPROVED (confident): Project 1 wrong, Project 2 correct")
keep = np.concatenate([cand, [test_df.index[test_df["id"] == "test_7057"][0]]])
test_df.loc[keep, ["id", "text_p2", "label"]].assign(
    p1=p1_pred[keep], ensemble=ens_pred[keep], ens_proba=ens_proba[keep].round(2)
).to_csv("results_p2/improved_examples.csv", index=False)

both_wrong = np.where((p1_pred != y_test) & (ens_pred != y_test) & (test_df["n_words"].values < 130))[0]
remaining = both_wrong[np.argsort(-np.abs(ens_proba[both_wrong] - 0.5))][:3]
show(remaining, "REMAINING ERRORS: both systems wrong")
print(f"\nBoth systems wrong: {((p1_pred != y_test) & (ens_pred != y_test)).sum():,}")
test_df.loc[remaining, ["id", "text_p2", "label"]].assign(
    ensemble=ens_pred[remaining]).to_csv("results_p2/remaining_errors.csv", index=False)

# ----- Prediction functions (also used by the app) -----
def predict_p1(text):
    return classifier.predict_proba(vectorizer.transform([preprocess(text)]))[0, 1]

def predict_p2(text):
    o = sentiment_model(light_clean(text), truncation=True, max_length=512)[0]
    p_tf = o["score"] if o["label"] == "POSITIVE" else 1 - o["score"]
    return best_w * p_tf + (1 - best_w) * predict_p1(text)

old_demo = [
    "I didn't expect much, but it did not disappoint. One of the best films this year!",
    "Great cast, beautiful visuals, but the story was boring and way too long.",
    "Oh wonderful, another sequel nobody asked for. Just brilliant.",
    "I'd give it 3/10.",
    "The movie was not good at all, very disappointing.",
]
old_demo_df = pd.DataFrame([{
    "review": t,
    "Project 1": f"{'Positive' if p1 >= .5 else 'Negative'} ({p1:.2f})",
    "Project 2": f"{'Positive' if p2 >= .5 else 'Negative'} ({p2:.2f})"}
    for t in old_demo for p1, p2 in [(predict_p1(t), predict_p2(t))]])
old_demo_df.to_csv("results_p2/old_demo_p1_vs_p2.csv", index=False)
print("\nPROJECT 1 DEMO REVIEWS, BOTH SYSTEMS:\n", old_demo_df.to_string(index=False))

# 8. Final unseen examples (12)
final_examples = [
    ("An unforgettable masterpiece with breathtaking cinematography and a moving score.", 1, "clear positive"),
    ("Two hours of my life I will never get back.", 0, "idiom, no opinion words"),
    ("The first half is slow, but the last thirty minutes are absolutely thrilling.", 1, "contrast with but"),
    ("I wanted to love it, but the characters were flat and the dialogue was painful.", 0, "contrast with but"),
    ("Not the worst film I've seen this year, but definitely not good either.", 0, "double negation"),
    ("The kids loved it and honestly so did I.", 1, "no movie words"),
    ("Visually stunning, yet emotionally empty.", 0, "mixed, short"),
    ("Wow, what a great idea to spend 200 million dollars on explosions and forget the story.", 0, "sarcasm"),
    ("8/10, would watch again.", 1, "written rating"),
    ("Terrifying, disturbing and full of blood: exactly what a horror fan wants.", 1, "horror vocabulary"),
    ("The acting was wooden and the plot predictable, but I couldn't stop smiling. A guilty pleasure.", 1, "so bad it's good"),
    ("Meryl Streep carries the whole film; without her it would be unwatchable.", 1, "praise inside criticism"),
]
rows = []
for text, expected, test in final_examples:
    p1, p2 = predict_p1(text), predict_p2(text)
    a = analyze_aspects(text)
    asp_str = ", ".join(f"{x}{'+' if o == 'Positive' else '-'}" for x, o in zip(a["aspect"], a["opinion"])) if not a.empty else "-"
    rows.append({"review": text, "tests": test, "expected": "Pos" if expected else "Neg",
                 "P1": f"{'Pos' if p1 >= .5 else 'Neg'} {p1:.2f}", "P2": f"{'Pos' if p2 >= .5 else 'Neg'} {p2:.2f}",
                 "P1 ok": "yes" if (p1 >= .5) == bool(expected) else "no",
                 "P2 ok": "yes" if (p2 >= .5) == bool(expected) else "no",
                 "aspects": asp_str})
final_df = pd.DataFrame(rows)
final_df.to_csv("results_p2/final_unseen_examples.csv", index=False)
print("\nFINAL UNSEEN EXAMPLES:\n", final_df.drop(columns="review").to_string())
print(f"Project 1 correct: {(final_df['P1 ok'] == 'yes').sum()} / {len(final_df)} | "
      f"Project 2 correct: {(final_df['P2 ok'] == 'yes').sum()} / {len(final_df)}")

# 9. Integrated application (Gradio) + notebook fallback
import gradio as gr
import html as html_lib
import inspect

def analyze_review(text):
    """Plain version (tables), used by the notebook demo."""
    if not text or len(text.split()) < 3:
        return "Please write a review of at least a few words.", None, None
    p1, p2 = predict_p1(text), predict_p2(text)
    verdict = "Positive" if p2 >= 0.5 else "Negative"
    conf = p2 if p2 >= 0.5 else 1 - p2
    md = (f"## {verdict}  ({conf:.0%} confidence)\n\n| Model | P(positive) |\n|---|---|\n"
          f"| Project 1: Bag of Words | {p1:.2f} |\n| Project 2: Ensemble (final) | {p2:.2f} |")
    aspects = analyze_aspects(text)
    aspects = (pd.DataFrame({"aspect": ["(no aspect found)"], "opinion": [""], "sentence": [""]})
               if aspects.empty else aspects[["aspect", "opinion", "sentence"]])
    similar = semantic_search(light_clean(text), k=3)[["similarity", "label", "review"]]
    return md, aspects, similar

def full_demo(text):
    """Notebook version of the app, in case the web interface is not available."""
    md, asp_df, sim_df = analyze_review(text)
    print(md.replace("## ", "").replace("|---|---|\n", ""))
    print("\nASPECTS:\n", asp_df.to_string(index=False))
    print("\nMOST SIMILAR REVIEWS:\n", sim_df.to_string(index=False))

print("\nNOTEBOOK DEMO:")
full_demo("Tom Hanks was brilliant, but the ending felt rushed and the music was forgettable.")

# ----- HTML pieces for the web app -----
esc = html_lib.escape

def card(title, note, body):
    note_html = f'<span>{note}</span>' if note else ""
    return f'<section class="mr-card"><header class="mr-card-head"><h3>{title}</h3>{note_html}</header>{body}</section>'

def sentiment_card(p1, p2):
    pos = p2 >= 0.5
    conf = p2 if pos else 1 - p2
    def bar(label, p, cls):
        return (f'<div class="mr-bar"><div class="mr-bar-top"><span>{label}</span><b>{p:.2f}</b></div>'
                f'<div class="mr-track"><div class="mr-fill {cls}" style="width:{max(p, 0.02) * 100:.0f}%"></div></div></div>')
    body = (f'<div class="mr-verdict">'
            f'<div><div class="mr-big {"pos" if pos else "neg"}">{"Positive" if pos else "Negative"}</div>'
            f'<p class="mr-sub">{conf:.0%} confidence from the final model</p></div>'
            f'<div class="mr-bars"><p class="mr-sub">Probability that the review is positive</p>'
            f'{bar("Project 1 · Bag of Words", p1, "old")}{bar("Project 2 · Ensemble (final)", p2, "new")}</div></div>')
    return card("Sentiment", "", body)

def aspect_cards(df):
    if df.empty:
        return card("Aspects", "", '<p class="mr-sub">No comment on acting, story, music, directing or visuals was found in this review.</p>')
    items = ""
    for _, r in df.iterrows():
        cls = "pos" if r["opinion"] == "Positive" else "neg"
        items += (f'<div class="mr-aspect"><div class="mr-aspect-top"><span class="mr-aspect-name">{esc(r["aspect"])}</span>'
                  f'<span class="mr-pill {cls}">{r["opinion"]}</span></div>'
                  f'<p class="mr-quote">“{esc(r["sentence"])}”</p></div>')
    return card("Aspects", "What the writer liked and disliked", f'<div class="mr-aspects">{items}</div>')

def ranked_list(df, title, note):
    rows = ""
    for i, r in enumerate(df.itertuples(), 1):
        cls = "pos" if r.label == "Positive" else "neg"
        rows += (f'<div class="mr-row"><div class="mr-rank">{i}</div>'
                 f'<div class="mr-row-body"><div class="mr-row-meta"><span class="mr-pill {cls}">{r.label}</span>'
                 f'<span class="mr-id">{esc(str(r.id))}</span></div>'
                 f'<p class="mr-review">{esc(r.full_text)}</p></div>'
                 f'<div class="mr-score"><b>{r.similarity:.2f}</b><span>similarity</span></div></div>')
    return card(title, note, f'<div class="mr-list">{rows}</div>')

def search_df(query, k):
    q = embedder.encode([query], normalize_embeddings=True)[0]
    scores = train_emb @ q
    top = np.argsort(-scores)[:k]
    return pd.DataFrame({"id": train_df["id"].values[top], "similarity": scores[top],
                         "label": ["Positive" if l == 1 else "Negative" for l in train_df["label"].values[top]],
                         "full_text": train_df["text_p2"].values[top]})

EMPTY_ANALYZE = ('<div class="mr-empty"><p class="mr-empty-title">Write a review, then press Analyze</p>'
                 '<p class="mr-sub">You will see its sentiment, the aspects the writer commented on, '
                 'and the three most similar reviews from 25,000 IMDb reviews.</p></div>')
EMPTY_SEARCH = ('<div class="mr-empty"><p class="mr-empty-title">Search by meaning</p>'
                '<p class="mr-sub">Describe what you are looking for in your own words, for example '
                '“the ending made me cry” or “cheap special effects”.</p></div>')

def ui_analyze(text):
    if not text or len(text.split()) < 3:
        return ('<div class="mr-empty"><p class="mr-empty-title">The review is too short</p>'
                '<p class="mr-sub">Write at least a few words.</p></div>'), "", ""
    p1, p2 = predict_p1(text), predict_p2(text)
    return (sentiment_card(p1, p2), aspect_cards(analyze_aspects(text)),
            ranked_list(search_df(light_clean(text), 3), "Similar reviews", "From 25,000 training reviews"))

def ui_search(query, k):
    if not query or not query.strip():
        return EMPTY_SEARCH
    return ranked_list(search_df(query, int(k)), "Results", f"Closest in meaning to “{esc(query)}”")

def models_ranking():
    rows = ""
    for i, r in enumerate(comparison.sort_values("Accuracy", ascending=False).itertuples(), 1):
        rows += (f'<div class="mr-row"><div class="mr-rank">{i}</div>'
                 f'<div class="mr-row-body"><p class="mr-model">{esc(r.Representation)}</p>'
                 f'<p class="mr-sub">{esc(r.Version)} · precision {r.Precision:.3f} · recall {r.Recall:.3f} · F1 {r.F1:.3f}</p></div>'
                 f'<div class="mr-score"><b>{r.Accuracy * 100:.2f}%</b><span>accuracy</span></div></div>')
    return card("Model comparison", "Same 25,000 unseen test reviews", f'<div class="mr-list">{rows}</div>')

SIDEBAR = f"""
<aside class="mr-side">
  <h2>Project overview</h2>
  <div class="mr-stat"><span>Final accuracy</span><b class="accent">{accuracy_score(y_test, ens_pred) * 100:.2f}%</b></div>
  <div class="mr-stat"><span>Project 1 baseline</span><b>{accuracy_score(y_test, p1_pred) * 100:.2f}%</b></div>
  <div class="mr-stat"><span>Reviews indexed</span><b>{len(train_df):,}</b></div>
  <div class="mr-stat"><span>Aspects tracked</span><b>5</b></div>
  <p class="mr-note">Accuracy is measured on the official IMDb test set, which the models never saw during training.</p>
</aside>"""

HEADER = '<div class="mr-header"><h1 class="mr-title">Movie Review Intelligence</h1></div>'

CSS = """
@import url('https://fonts.googleapis.com/css2?family=Archivo:wdth,wght@100,500;100,700;112,800&family=IBM+Plex+Sans:wght@400;500;600&display=swap');
:root, .dark {
  --bg:#0f0f11; --panel:#17171b; --panel-2:#1f1f25; --line:#2c2c33;
  --text:#f4f4f5; --muted:#b4b4be; --faint:#8e8e99;
  --accent:#ff6a13; --accent-hover:#ff8240; --pos:#4cc79a; --neg:#ff6b6b;
  --s1:4px; --s2:8px; --s3:12px; --s4:16px; --s5:24px; --s6:32px; --s7:48px;
  --r:16px; --r-sm:12px;
  --display:'Archivo', 'Arial Narrow', Arial, sans-serif;
  --body:'IBM Plex Sans', 'Segoe UI', Arial, sans-serif;
}
body, .gradio-container { background:var(--bg) !important; font-family:var(--body) !important; color:var(--text); }
.gradio-container { max-width:1200px !important; padding:var(--s7) var(--s5) !important; }
footer { display:none !important; }

/* layout */
#mr-shell { gap:var(--s6) !important; align-items:flex-start !important; }
#mr-sidebar { flex:0 0 260px !important; max-width:260px; min-width:0 !important; position:sticky; top:var(--s5);
  background:var(--panel); border:1px solid var(--line); border-radius:var(--r); padding:var(--s5) !important; }
#mr-main { flex:1 1 0 !important; min-width:0 !important; gap:var(--s5) !important; }
#mr-main .block, #mr-main .form { background:transparent !important; border:none !important; padding:0 !important; box-shadow:none !important; }
#mr-main .html-container, #mr-main .prose { padding:0 !important; }
#mr-main .tabitem, #mr-main .tabitem > .column { gap:var(--s5) !important; }

/* sidebar */
.mr-side h2 { font-family:var(--display); font-size:18px; font-weight:700; margin:0 0 var(--s3); color:var(--text); }
.mr-stat { display:flex; justify-content:space-between; align-items:baseline; gap:var(--s3);
  padding:var(--s3) 0; border-top:1px solid var(--line); }
.mr-stat span { color:var(--muted); font-size:14px; }
.mr-stat b { font-family:var(--display); font-size:20px; font-weight:700; color:var(--text); font-variant-numeric:tabular-nums; }
.mr-stat b.accent { color:var(--accent); }
.mr-note { margin:var(--s3) 0 0; color:var(--faint); font-size:13px; line-height:1.55; }

/* header and tabs */
.mr-header { text-align:center; padding:0 0 var(--s2); }
.mr-title { font-family:var(--display); font-stretch:112%; font-weight:800; font-size:40px; line-height:1.1;
  letter-spacing:-0.01em; margin:0; color:var(--text); }
#mr-main .tab-wrapper, #mr-main .tab-container { justify-content:center !important; border-color:var(--line) !important; }
#mr-main .tab-wrapper { padding-bottom:var(--s3); margin-bottom:var(--s2); }
#mr-main button[role="tab"] { border-radius:999px !important; padding:var(--s2) var(--s5) !important; font-size:15px !important;
  font-weight:500 !important; color:var(--muted) !important; border:none !important; }
#mr-main button[role="tab"]:hover { color:var(--text) !important; }
#mr-main button[role="tab"][aria-selected="true"], #mr-main .tab-container button.selected {
  background:var(--accent) !important; color:#141414 !important; font-weight:600 !important; }
#mr-main .tab-container button.selected::after { display:none !important; }

/* inputs and buttons */
#mr-main textarea, #mr-main input[type=text] { background:var(--panel) !important; color:var(--text) !important;
  border:1px solid var(--line) !important; border-radius:var(--r) !important; padding:var(--s4) !important;
  font-size:16px !important; line-height:1.6 !important; }
#mr-main textarea:focus, #mr-main input[type=text]:focus { border-color:var(--accent) !important;
  box-shadow:0 0 0 3px rgba(255,106,19,.25) !important; outline:none !important; }
#mr-main textarea::placeholder, #mr-main input::placeholder { color:var(--faint) !important; }
.mr-actions { justify-content:flex-end !important; }
.mr-go { background:var(--accent) !important; color:#141414 !important; font-weight:600 !important; font-size:15px !important;
  border:none !important; border-radius:var(--r-sm) !important; padding:var(--s3) var(--s6) !important; transition:background .15s, transform .05s; }
.mr-go:hover { background:var(--accent-hover) !important; }
.mr-go:active { transform:translateY(1px); }
.mr-go:focus-visible { outline:2px solid var(--text) !important; outline-offset:2px; }
#mr-main .gallery-item { background:var(--panel) !important; border:1px solid var(--line) !important; color:var(--muted) !important;
  border-radius:999px !important; padding:var(--s2) var(--s4) !important; font-size:14px !important; }
#mr-main .gallery-item:hover { border-color:var(--accent) !important; color:var(--text) !important; }
#mr-main label span, #mr-main .label-wrap span { color:var(--faint) !important; font-size:13px !important; }

/* cards */
.mr-card { background:var(--panel); border:1px solid var(--line); border-radius:var(--r); padding:var(--s5); color:var(--text); }
.mr-card-head { display:flex; justify-content:space-between; align-items:baseline; gap:var(--s4); margin-bottom:var(--s4); }
.mr-card-head h3 { font-family:var(--display); font-size:20px; font-weight:700; margin:0; color:var(--text); }
.mr-card-head span { color:var(--faint); font-size:13px; text-align:right; }
.mr-sub { color:var(--muted); font-size:14px; line-height:1.5; margin:0; }
.mr-empty { border:1px dashed var(--line); border-radius:var(--r); padding:var(--s7) var(--s5); text-align:center; }
.mr-empty-title { font-family:var(--display); font-size:18px; font-weight:700; margin:0 0 var(--s2); color:var(--text); }

/* sentiment */
.mr-verdict { display:grid; grid-template-columns:minmax(0,1fr) minmax(0,1.3fr); gap:var(--s6); align-items:center; }
.mr-big { font-family:var(--display); font-stretch:112%; font-size:48px; font-weight:800; line-height:1; margin-bottom:var(--s2); }
.mr-big.pos { color:var(--pos); } .mr-big.neg { color:var(--neg); }
.mr-bars { display:flex; flex-direction:column; gap:var(--s4); }
.mr-bar-top { display:flex; justify-content:space-between; font-size:14px; color:var(--muted); margin-bottom:var(--s2); }
.mr-bar-top b { color:var(--text); font-variant-numeric:tabular-nums; }
.mr-track { height:8px; border-radius:999px; background:var(--panel-2); overflow:hidden; }
.mr-fill { height:100%; border-radius:999px; }
.mr-fill.old { background:#77777f; } .mr-fill.new { background:var(--accent); }

/* aspects */
.mr-aspects { display:grid; grid-template-columns:repeat(auto-fill, minmax(220px, 1fr)); gap:var(--s3); }
.mr-aspect { background:var(--panel-2); border-radius:var(--r-sm); padding:var(--s4); display:flex; flex-direction:column; gap:var(--s2); }
.mr-aspect-top { display:flex; justify-content:space-between; align-items:center; gap:var(--s2); }
.mr-aspect-name { font-weight:600; font-size:16px; }
.mr-quote { color:var(--muted); font-size:14px; line-height:1.5; margin:0; }
.mr-pill { font-size:12px; font-weight:600; padding:2px 10px; border-radius:999px; white-space:nowrap; }
.mr-pill.pos { background:rgba(76,199,154,.15); color:var(--pos); }
.mr-pill.neg { background:rgba(255,107,107,.15); color:var(--neg); }

/* ranked lists */
.mr-row { display:grid; grid-template-columns:48px minmax(0,1fr) auto; gap:var(--s5); align-items:center;
  padding:var(--s4) 0; border-top:1px solid var(--line); }
.mr-list .mr-row:first-child { border-top:none; padding-top:0; }
.mr-list .mr-row:last-child { padding-bottom:0; }
.mr-rank { font-family:var(--display); font-size:32px; font-weight:800; color:var(--accent); text-align:center; font-variant-numeric:tabular-nums; }
.mr-row-meta { display:flex; gap:var(--s2); align-items:center; margin-bottom:var(--s2); }
.mr-id { font-size:12px; color:var(--faint); }
.mr-review { margin:0; font-size:15px; line-height:1.6; color:#dcdce1;
  display:-webkit-box; -webkit-line-clamp:3; -webkit-box-orient:vertical; overflow:hidden; }
.mr-model { margin:0 0 var(--s1); font-weight:600; font-size:16px; color:var(--text); }
.mr-score { text-align:right; display:flex; flex-direction:column; }
.mr-score b { font-family:var(--display); font-size:26px; font-weight:700; font-variant-numeric:tabular-nums; }
.mr-score span { color:var(--faint); font-size:12px; }

@media (max-width: 860px) {
  .gradio-container { padding:var(--s5) var(--s4) !important; }
  #mr-shell { flex-direction:column !important; }
  #mr-sidebar { position:static; flex:1 1 auto !important; max-width:none; width:100%; }
  .mr-title { font-size:30px; }
  .mr-verdict { grid-template-columns:1fr; gap:var(--s5); }
  .mr-row { grid-template-columns:36px minmax(0,1fr); gap:var(--s4); }
  .mr-score { grid-column:2; text-align:left; flex-direction:row; gap:var(--s2); align-items:baseline; }
  .mr-rank { font-size:26px; }
}
"""

THEME = gr.themes.Base(primary_hue="orange", neutral_hue="zinc").set(
    body_background_fill="#0f0f11", body_background_fill_dark="#0f0f11",
    body_text_color="#f4f4f5", body_text_color_dark="#f4f4f5",
    block_background_fill="#17171b", block_background_fill_dark="#17171b",
    block_border_color="#2c2c33", block_border_color_dark="#2c2c33",
    border_color_primary="#2c2c33", border_color_primary_dark="#2c2c33",
    input_background_fill="#17171b", input_background_fill_dark="#17171b",
    button_primary_background_fill="#ff6a13", button_primary_background_fill_dark="#ff6a13",
    button_primary_text_color="#141414", button_primary_text_color_dark="#141414",
    slider_color="#ff6a13", slider_color_dark="#ff6a13",
)

app_examples = [
    "Great cast, beautiful visuals, but the story was boring and way too long.",
    "Meryl Streep was outstanding and the soundtrack is gorgeous. A must see!",
    "The acting was wooden and the plot predictable, but I couldn't stop smiling. A guilty pleasure.",
    "The plot made no sense and the special effects looked cheap. I'd give it 3/10.",
]

# css/theme go to Blocks in older Gradio and to launch() in newer versions
style = {"css": CSS, "theme": THEME}
in_blocks = "css" in inspect.signature(gr.Blocks.__init__).parameters

with gr.Blocks(title="Movie Review Intelligence", **(style if in_blocks else {})) as app:
    with gr.Row(elem_id="mr-shell"):
        with gr.Column(elem_id="mr-sidebar"):
            gr.HTML(SIDEBAR)
        with gr.Column(elem_id="mr-main"):
            gr.HTML(HEADER)
            with gr.Tabs():
                with gr.Tab("Analyze"):
                    review_in = gr.Textbox(lines=5, show_label=False, placeholder="Write a movie review in English…")
                    with gr.Row(elem_classes=["mr-actions"]):
                        analyze_btn = gr.Button("Analyze review", elem_classes=["mr-go"], scale=0, min_width=200)
                    gr.Examples(examples=app_examples, inputs=review_in, label="Try an example")
                    sentiment_out = gr.HTML(EMPTY_ANALYZE)
                    aspects_out = gr.HTML()
                    similar_out = gr.HTML()
                    analyze_btn.click(ui_analyze, inputs=review_in, outputs=[sentiment_out, aspects_out, similar_out])
                with gr.Tab("Search"):
                    with gr.Row():
                        query_in = gr.Textbox(show_label=False, placeholder="Search reviews…", scale=5)
                        search_btn = gr.Button("Search", elem_classes=["mr-go"], scale=0, min_width=140)
                    k_in = gr.Slider(1, 10, value=5, step=1, label="Number of results")
                    search_out = gr.HTML(EMPTY_SEARCH)
                    search_btn.click(ui_search, inputs=[query_in, k_in], outputs=search_out)
                    query_in.submit(ui_search, inputs=[query_in, k_in], outputs=search_out)
                with gr.Tab("Models"):
                    gr.HTML(models_ranking())

app.launch(share=True, **({} if in_blocks else style))
